# Module 04 - Full RAG Pipeline

**Duration:** 60 minutes

This module wires all the pieces together into a working system.
By the end you will have a RAG pipeline running against the sample documents,
a Gradio GUI to interact with it, and an understanding of how the LLM parameters
affect the quality and style of the answers.

---


## 4.1 Setting up the RAGTool

The `RAGTool` class in `src/ragsst/ragtool.py` is the central object in this repo.
It wraps the vector store, the LLM connection, and all the retrieval logic.

Before running the cells below, make sure Ollama is running:

```bash
ollama serve
```


In [ ]:
from ragsst.ragtool import RAGTool
import ragsst.parameters as p

print('Available embedding models:', p.EMBEDDING_MODELS)
print('Available LLMs:', p.LLM_CHOICES)


In [ ]:
tool = RAGTool(
    model='llama3.2',
    data_path='../data/sample_docs',
    collection_name='workshop_docs',
)

# This ingests documents if the collection is empty, or loads it if it already exists
tool.setup_vec_store()

print(f'Collection: {tool.collection_name}')
print(f'Documents in collection: {tool.collection.count()}')


## 4.2 Retrieval

Let us look at what the retriever actually returns before we involve the LLM.
This is worth doing separately because retrieval failures are often the root cause
of bad answers.


In [ ]:
query = 'What services does the AI service center offer?'

# get_relevant_text returns plain text, concatenated
context = tool.get_relevant_text(query, nresults=3, sim_th=0.3)
print('Retrieved context:')
print('-' * 60)
print(context)


In [ ]:
# retrieve_with_metadata shows similarity scores and sources
context_with_meta = tool.retrieve_with_metadata(query, nresults=3, sim_th=0.3)
print(context_with_meta)


The similarity score tells you how confident the retriever is.
A score below 0.3 usually means the question is about something not in the documents.
That is when you want the system to say 'I don't know' rather than hallucinate.


## 4.3 Prompt construction

The prompt is how we communicate the retrieved context to the LLM.
Good prompt design matters a lot here.


In [ ]:
context = tool.get_relevant_text(query, nresults=3)
prompt = tool.get_context_prompt(query, context)

print('Full prompt sent to LLM:')
print('=' * 60)
print(prompt)
print('=' * 60)


Read the prompt carefully. You can see exactly what the model is working from.
If the context does not contain the answer, the model should not be able to answer correctly.
When it does anyway, that is a hallucination.


## 4.4 Generation

Now we add the LLM step.


In [ ]:
# Single-turn RAG query
answer = tool.rag_query(
    user_msg=query,
    sim_th=0.3,
    nresults=3,
    top_k=5,
    top_p=0.9,
    temp=0.3,
)

print(f'Question: {query}')
print(f'\nAnswer: {answer}')


In [ ]:
# Try a few different questions
questions = [
    'Who is Sherlock Holmes?',
    'What happens at the end of Die Hard?',
    'What is the capital of Australia?',  # not in the documents
]

for q in questions:
    answer = tool.rag_query(q, sim_th=0.3, nresults=3, top_k=5, top_p=0.9, temp=0.3)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print()


The last question is not in the documents. What does the model do?
The answer depends on the similarity threshold. With a high threshold,
the retriever returns nothing and the model is told so.
With a low threshold, it retrieves something weakly related and may hallucinate.


## 4.5 LLM parameters

Three parameters control the randomness of the generated text.

**Temperature** scales the probability distribution of the next token.
Low temperature (0.1-0.3) makes the model more deterministic and focused.
High temperature (0.7-1.0) makes it more creative and varied.
For factual RAG, keep it low.

**Top-k** limits sampling to the k most probable tokens at each step.
A value of 1 is greedy (always picks the most likely token).
Values of 5-40 are typical.

**Top-p** (nucleus sampling) keeps the smallest set of tokens whose cumulative
probability exceeds p. It adapts to the entropy of the distribution.
0.9 is a common default.


In [ ]:
# Compare different temperature settings on the same question
q = 'What happened to Hans Gruber at the end of the film?'

for temp in [0.1, 0.5, 0.9]:
    answer = tool.rag_query(q, sim_th=0.3, nresults=3, top_k=10, top_p=0.9, temp=temp)
    print(f'temp={temp}: {answer[:200]}')
    print()


## 4.6 Prompt engineering

The system prompt and instruction phrasing affect the output as much as the parameters.


In [ ]:
import requests, json
from os import getenv
from urllib.parse import urljoin

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')


def generate_with_prompt(prompt: str, temp: float = 0.3) -> str:
    r = requests.post(
        OLLAMA_URL + '/generate',
        json={'model': 'llama3.2', 'prompt': prompt, 'stream': False,
              'options': {'temperature': temp}}
    )
    return json.loads(r.text).get('response', '')


context = tool.get_relevant_text('What is the AI service center?', nresults=2)

prompt_v1 = f'Context:\n{context}\n\nQuestion: What is the AI service center?\nAnswer:'

prompt_v2 = (
    'You are a helpful assistant. Use only the context below to answer. '
    'If the answer is not in the context, say so.\n\n'
    f'Context:\n{context}\n\n'
    'Question: What is the AI service center?\n'
    'Answer in one sentence:'
)

print('Prompt v1 answer:')
print(generate_with_prompt(prompt_v1))
print('\nPrompt v2 answer:')
print(generate_with_prompt(prompt_v2))


## 4.7 The Gradio GUI

The repo includes a full web interface. Run the cell below to launch it.
You can also run it from the terminal: `uv run python local-rag-gui.py`


In [ ]:
from ragsst.interface import make_interface

gui = make_interface(tool)
gui.launch()


---

**Exercises**

1. Ask a question that requires information from two different documents
   (e.g. 'What do Die Hard and Sherlock Holmes have in common?').
   Set nresults=4. Does the answer draw from both?

2. Set sim_th=0.8 and ask a question. What changes? Lower it to 0.1.
   Where does the threshold need to be to get useful answers on your questions?

3. Write a system prompt that tells the model to always answer in Spanish.
   Does it comply? What happens when the context is in English?

---

**Further reading**

- Ollama API reference: https://github.com/ollama/ollama/blob/main/docs/api.md
- Gradio docs: https://www.gradio.app/docs
- Prompt engineering guide: https://www.promptingguide.ai/
